# Пакет `epi`: от папки с файлами `.e` до обучающей выборки

Демонстрация четырёх задач пайплайна:

1. **Каталог** — обойти папки с записями и собрать метаданные; краулер отслеживает изменения на диске.
2. **Данные пациента** — достать записи, метаданные, сигнал и маркеры конкретного человека.
3. **Обучающая выборка** — нарезать сигнал на окна и получить сбалансированный набор.
4. **Статистика** — выжимка по всей коллекции.

### Как запустить

Ноутбук должен работать на том же интерпретаторе, где стоит пакет. Проще всего запустить Jupyter прямо из виртуального окружения — тогда ядро `Python 3` и будет нужным интерпретатором:

```bash
cd /home/www1rt/Documents/EPI
./.venv/bin/jupyter lab demo.ipynb
```

Если вы предпочитаете запускать Jupyter из другого места, зарегистрируйте ядро один раз и выберите его в меню *Kernel → Change kernel*:

```bash
./.venv/bin/python -m ipykernel install --user --name epi --display-name "EPI (.venv)"
```

In [ ]:
import sys

import pandas as pd

import epi
from epi import Catalog, WindowSet, plot_marker, plot_window, render, summary

pd.set_option("display.width", 200, "display.max_colwidth", 60)

print("интерпретатор:", sys.executable)
print("версия epi:  ", epi.__version__)

## Задача 1. Каталог и краулер

`Catalog` — база SQLite с метаданными всей коллекции. Метод `sync()` обходит папку, разбирает новые файлы `.e` и приводит каталог в соответствие с диском.

Обход дешёвый: файл разбирается заново только если изменились его размер или время правки, **и** при этом разошёлся отпечаток содержимого. Поэтому повторный `sync()` на неизменных данных практически ничего не делает.

In [ ]:
catalog = Catalog("catalog.sqlite")

# первый вызов разбирает файлы, последующие только сверяют их с диском
counts = catalog.sync(".")
catalog

Все методы выборки возвращают списки словарей, поэтому их можно сразу передать в `pandas.DataFrame`.

Обратите внимание на `recordings` и `hours`: пациентов 16, а файлов 18 — у части людей несколько записей. Опознание идёт по GUID из заголовка файла, а не по имени папки, потому что имена папок содержат номера выгрузок. Одна и та же подпись `Patient1` на деле покрывала четырёх разных людей.

In [ ]:
patients = pd.DataFrame(catalog.patients())
print(catalog.totals())
patients[["patient_key", "alt_id", "dob", "recordings", "hours", "seizures"]].head(8)

## Задача 2. Данные и метаданные пациента

`catalog.patient()` собирает всё об одном человеке: поля из заголовка, список записей и все его приступы. Ключ можно указать началом GUID.

In [ ]:
key = patients.loc[0, "patient_key"]
info = catalog.patient(key)

print("пациент:       ", info["patient_key"])
print("дата рождения: ", info["dob"])
print("заметки врача: ", info["notes"])
print("приступов:     ", len(info["seizures"]))

pd.DataFrame(info["recordings"])[
    ["folder", "sampling_rate", "n_channels", "n_segments", "duration_sec"]
]

### От каталога к сигналу

`catalog.open()` возвращает `NicoletEReader` — это мост от строки каталога к сырым данным. Чтение оконное: с диска поднимается только запрошенный интервал, поэтому минута из многочасовой записи стоит несколько мегабайт.

Важная особенность формата: запись состоит из **сегментов**, разделённых настоящими разрывами во времени. Отсчёты читаются только внутри одного сегмента, поэтому `start_sec` отсчитывается от начала текущего сегмента.

In [ ]:
reader = catalog.open(info["recordings"][0]["id"])

print(reader)
print("каналы:  ", ", ".join(reader.channels))
print("сегменты:", reader.segments[:3])

# один канал за первую минуту, в микровольтах
ekg = reader.read_channel("EKG", start_sec=0, duration_sec=60)
print("\nEKG:", ekg.shape, ekg.dtype)

# несколько каналов сразу: массив (отсчёты, каналы)
block = reader.read(["Fp1", "Fp2", "C3", "C4"], start_sec=0, duration_sec=10)
print("блок из 4 каналов:", block.shape)

### Маркеры врача

Маркеры собираются из двух библиотек сразу, потому что каждая ошибается по-своему: `neo` верно раскладывает их по сегментам, но многие типы оставляет как `UNKNOWN` и портит кириллицу, а `pynicolet` называет больше типов и правильно декодирует текст, но в многосегментных файлах прижимает маркеры к концу записи. Ридер берёт от каждой то, что она делает верно.

Колонки `type_neo` и `type_pynicolet` показывают, что сказала каждая библиотека по отдельности — по ним видно, откуда взялось итоговое название.

In [ ]:
events = pd.DataFrame(reader.read_events())
print("маркеров в записи:", len(events))

events[["type", "text", "user", "segment", "segment_time_sec", "duration_sec",
        "type_neo", "type_pynicolet"]].head(10)

In [ ]:
# маркеры по всей коллекции, с фильтром по типу
seizures = catalog.events(type="Seizure")
print("приступов в коллекции:", len(seizures))

# красная линия — начало маркера, полоса — его длительность
plot_marker(catalog, seizures[0], pad_sec=5, window_sec=25);

## Задача 3. Обучающая выборка

`WindowSet` нарезает сигнал на окна одинаковой формы. Три решения продиктованы свойствами данных:

- окно целиком лежит **внутри одного сегмента** и не пересекает границу, потому что между сегментами настоящий разрыв во времени;
- разбиение делается **по пациентам**: окна одной записи перекрываются во времени и тривиально предсказываются друг из друга, поэтому случайное разбиение самих окон протащило бы тест в обучение;
- пациенты раздаются от самых богатых приступами к бедным, иначе фолд легко остаётся вовсе без положительных примеров.

In [ ]:
windows = WindowSet(catalog, length_sec=10.0, stride_sec=5.0, min_overlap_sec=5.0)

windows.build()   # разметить окна
print()
windows.split()   # раздать пациентов по фолдам
pd.DataFrame(windows.folds())

Приступы занимают около 3% сигнала, поэтому выборка набирается по классам отдельно, каждый до своей квоты. Заодно отбрасываются окна с плоским сигналом: усечённые выгрузки `Pruned` добиты сплошными нулями, и такие куски не должны попасть в обучение.

Сигнал приводится к 19 электродам схемы 10-20, которые есть в каждой записи, и к общей частоте 256 Гц — в коллекции есть и 500, и 512 Гц.

In [ ]:
x, y = windows.sample("train", n=64, positive_ratio=0.5, seed=0)

print("x:", x.shape, x.dtype, " (окна, каналы, отсчёты)")
print("y:", y.shape, " приступов:", int(y.sum()))
print("каналы:", ", ".join(windows.channels))

In [ ]:
# посмотреть глазами на то, что уехало в обучение
first_positive = int(y.argmax())
plot_window(x[first_positive], windows.channels, windows.rate, label=int(y[first_positive]));

In [ ]:
# то же самое, но на диск: внутри .npz лежат x, y, channels и rate
windows.export("val", out_dir="training", n=64, seed=0)

## Задача 4. Статистика

`summary()` собирает отчёт по каталогу вложенным словарём, `render()` печатает его по-человечески. Всё считается по базе, поэтому стоит миллисекунды и не трогает файлы `.e`.

In [ ]:
report = summary(catalog)
render(report)

In [ ]:
# отдельные разделы отчёта удобно смотреть таблицей
print("протоколы съёмки:")
display(pd.DataFrame(report["protocols"]))

print("\nтипы маркеров:")
display(pd.Series(report["markers"], name="штук").to_frame().head(10))

In [ ]:
import matplotlib.pyplot as plt

lengths = [e["duration_sec"] or 0 for e in catalog.events(type="Seizure")]

figure, axes = plt.subplots(figsize=(9, 3.5))
axes.hist([v for v in lengths if v <= 120], bins=40, color="steelblue")
axes.set_xlabel("длительность приступа, с")
axes.set_ylabel("маркеров")
axes.set_title("Приступы короче двух минут (%d из %d)"
               % (sum(v <= 120 for v in lengths), len(lengths)))
axes.spines[["top", "right"]].set_visible(False)
figure.tight_layout()

## Обновление датасета: добавление и удаление

Краулер сравнивает диск с каталогом. Повторный `sync()` на неизменных данных ничего не разбирает — все записи попадают в `unchanged`.

Записи, исчезнувшие с диска, помечаются удалёнными, но не стираются: так обучение можно проследить до тех данных, на которых оно шло. Из выборок они при этом сразу исчезают. Если файл вернётся на место, он опознается по отпечатку содержимого и снова станет живым (`revived`).

In [ ]:
catalog.sync(".")   # ничего не изменилось -> всё в unchanged

## То же самое из командной строки

```bash
python -m epi sync .                        # обновить каталог
python -m epi stats --json digest.json      # статистика, заодно в JSON
python -m epi patient <ключ>                # всё об одном пациенте
python -m epi index                         # разметить окна
python -m epi split                         # раздать пациентов по фолдам
python -m epi export --fold train --n 2000  # сохранить выборку в .npz
python -m epi plot --type Seizure --n 3     # нарисовать маркеры в .png
```

## Что стоит решить перед обучением

- **Длина окна.** При 10 секундах 38 сегментов короче одного окна выпадают целиком, а медиана длительности приступа — 5.5 секунды.
- **Предобработка.** Сигнал отдаётся сырым, без фильтрации; фильтр и отбраковка артефактов в пакет не входят.
- **Размер валидации.** В `val` попали всего 2 пациента — для отбора модели это мало.
- **Аномалии разметки.** Самый долгий маркер приступа длится 1655 секунд (27 минут), что на приступ не похоже.